# MarketSense - Model A (baseline 5-session direction classifier)

**Inputs:** `model_a_dataset.parquet` + `model_a_meta.json` exported by `python -m scripts.build_model_a_dataset`.
**Target:** `target_direction_5d` = 1 if `forward_return_5d` > 0 else 0 (raw closes, 5 *market* sessions ahead).
**Features:** OHLCV-derived + RSI/MACD, all known at the post-market-close cutoff of `trading_date`.

Order: config -> load + validate -> chronological purged split -> majority benchmark -> logistic regression -> XGBoost -> compare -> save artifacts.
**The test set is used once, at the end of each model's fit. Do not tune on it.**

## 0. Setup

In [ ]:
import sys, subprocess, importlib
try:
    importlib.import_module("xgboost")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])

import json, platform, re, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, sklearn, xgboost as xgb
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 30)
print("pandas", pd.__version__, "| sklearn", sklearn.__version__, "| xgboost", xgb.__version__)

## 1. Configuration
`DATA_SOURCE`: **"drive"** (recommended: copy the two files to Google Drive once), **"upload"** (pick the two files in a dialog each session), or **"local"**.

In [ ]:
CFG = dict(
    DATA_SOURCE="drive",                                    # "drive" | "upload" | "local"
    DRIVE_DIR="/content/drive/MyDrive/MarketSense/model_a",  # folder holding the two exported files
    LOCAL_DIR="data/model_a",
    OUT_DIR="/content/model_a_artifacts",
    TRAIN_FRAC=0.70,                 # by unique DATES, chronological; the remainder after VAL is test
    VAL_FRAC=0.15,
    DROP_CORP_ACTION_SUSPECT=True,   # drop rows whose label window contains a raw |1-day move| above the export threshold
    CLIP_PCT=(0.5, 99.5),            # winsorise bounds (learned on TRAIN only) used for logistic regression
    BOOTSTRAP_REPS=300, BOOTSTRAP_BLOCK_DATES=10,
    SEED=42,
)
Path(CFG["OUT_DIR"]).mkdir(parents=True, exist_ok=True)

## 2. Load the exported dataset and validate it

In [ ]:
def get_data_dir() -> Path:
    if CFG["DATA_SOURCE"] == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        return Path(CFG["DRIVE_DIR"])
    if CFG["DATA_SOURCE"] == "upload":
        from google.colab import files
        up = files.upload()                       # select model_a_dataset.parquet AND model_a_meta.json
        d = Path("/content/model_a_upload"); d.mkdir(exist_ok=True)
        for name, data in up.items():
            (d / re.sub("[ ]?[(][0-9]+[)]", "", name)).write_bytes(data)   # strip browser (1) suffixes
        return d
    return Path(CFG["LOCAL_DIR"])

DATA_DIR = get_data_dir()
ds = pd.read_parquet(DATA_DIR / "model_a_dataset.parquet")
meta = json.loads((DATA_DIR / "model_a_meta.json").read_text())

FEATURES, TARGET, RET, HORIZON = meta["features"], meta["target_col"], meta["return_col"], meta["horizon_sessions"]
ds["trading_date"] = pd.to_datetime(ds["trading_date"]); ds["label_end_date"] = pd.to_datetime(ds["label_end_date"])

# ---- hard validation: stop here if anything is off ----
assert set(FEATURES) <= set(ds.columns), "feature columns missing from dataset"
assert not any(k in f for f in FEATURES for k in ("target", "forward", "label", "symbol", "stock_id", "date")), "outcome/identifier in features"
assert ds[FEATURES].notna().all().all() and np.isfinite(ds[FEATURES].to_numpy(float)).all(), "NaN/inf in features"
assert set(ds[TARGET].unique()) <= {0, 1}, "target must be 0/1"
assert ((ds[RET] > 0).astype(int) == ds[TARGET]).all(), "target inconsistent with forward return"
assert not ds.duplicated(["symbol", "trading_date"]).any(), "duplicate (symbol, date) rows"
assert (ds["label_end_date"] > ds["trading_date"]).all(), "label must end after prediction date"
print(f"rows={len(ds):,}  stocks={ds.symbol.nunique()}  dates={ds.trading_date.min().date()} -> {ds.trading_date.max().date()}")
print("features:", len(FEATURES), "| corp-action-suspect rows:", int(ds.corp_action_suspect_5d.sum()))
print("overall share up:", round(ds[TARGET].mean(), 4), "| exact-zero returns (labelled 0):", int((ds[RET] == 0).sum()))
if CFG["DROP_CORP_ACTION_SUSPECT"]:
    ds = ds[~ds.corp_action_suspect_5d].reset_index(drop=True)
    print("after dropping suspect rows:", f"{len(ds):,}")

## 3. Chronological split with purging (no overlap of 5-session label windows)
Split by **unique dates**, not rows, so a date never straddles two sets. A train row is kept only if its label window
(`label_end_date`) ends **before** validation starts; same for validation vs test. This removes the leakage that a plain
date split has when labels look 5 sessions into the future.

In [ ]:
dates = np.sort(ds["trading_date"].unique())
val_start  = pd.Timestamp(dates[int(len(dates) * CFG["TRAIN_FRAC"])])
test_start = pd.Timestamp(dates[int(len(dates) * (CFG["TRAIN_FRAC"] + CFG["VAL_FRAC"]))])

train = ds[(ds.trading_date < val_start) & (ds.label_end_date < val_start)]
val   = ds[(ds.trading_date >= val_start) & (ds.trading_date < test_start) & (ds.label_end_date < test_start)]
test  = ds[ds.trading_date >= test_start]

assert train.label_end_date.max() < val.trading_date.min(), "train labels overlap validation"
assert val.label_end_date.max() < test.trading_date.min(), "validation labels overlap test"

rows = []
for name, d in [("train", train), ("val", val), ("test", test)]:
    rows.append(dict(split=name, rows=len(d), stocks=d.symbol.nunique(), start=d.trading_date.min().date(),
                     end=d.trading_date.max().date(), share_up=round(d[TARGET].mean(), 4)))
print(pd.DataFrame(rows).to_string(index=False))
print("rows purged at the boundaries:", len(ds[ds.trading_date < test_start]) - len(train) - len(val))

X_tr, y_tr = train[FEATURES], train[TARGET].to_numpy()
X_va, y_va = val[FEATURES],   val[TARGET].to_numpy()
X_te, y_te = test[FEATURES],  test[TARGET].to_numpy()

## 4. Metrics helpers and the majority-class benchmark

In [ ]:
def metrics(y, proba, thr=0.5):
    pred = (np.asarray(proba) >= thr).astype(int)
    auc = roc_auc_score(y, proba) if len(np.unique(proba)) > 1 and len(np.unique(y)) > 1 else 0.5
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(thr=round(float(thr), 3), accuracy=accuracy_score(y, pred), balanced_acc=balanced_accuracy_score(y, pred),
                precision=precision_score(y, pred, zero_division=0), recall=recall_score(y, pred, zero_division=0),
                f1=f1_score(y, pred, zero_division=0), roc_auc=auc, tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def tune_threshold(y, proba):
    """Pick the threshold on VALIDATION that maximises balanced accuracy (ties -> closest to 0.5)."""
    grid = np.round(np.arange(0.35, 0.651, 0.005), 3)
    best = max(grid, key=lambda t: (round(balanced_accuracy_score(y, (proba >= t).astype(int)), 6), -abs(t - 0.5)))
    return float(best)

RESULTS = {}
maj = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
p_va, p_te = maj.predict_proba(X_va)[:, 1], maj.predict_proba(X_te)[:, 1]
RESULTS["majority_class"] = {"val": metrics(y_va, p_va), "test": metrics(y_te, p_te)}
print("majority class in TRAIN:", int(maj.classes_[np.argmax(np.bincount(y_tr))]), "| test share up:", round(y_te.mean(), 4))
pd.DataFrame(RESULTS["majority_class"]).T.round(4)

## 5. Baseline 1: logistic regression (winsorise -> impute -> scale -> LR, balanced class weights)

In [ ]:
lo, hi = np.percentile(X_tr, CFG["CLIP_PCT"][0], axis=0), np.percentile(X_tr, CFG["CLIP_PCT"][1], axis=0)   # TRAIN only
clip = lambda X: pd.DataFrame(np.clip(X.to_numpy(float), lo, hi), columns=FEATURES, index=X.index)

logreg = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000, random_state=CFG["SEED"]))
logreg.fit(clip(X_tr), y_tr)
lr_va, lr_te = logreg.predict_proba(clip(X_va))[:, 1], logreg.predict_proba(clip(X_te))[:, 1]
lr_thr = tune_threshold(y_va, lr_va)
RESULTS["logistic_regression"] = {"val": metrics(y_va, lr_va, lr_thr), "test": metrics(y_te, lr_te, lr_thr)}
pd.DataFrame(RESULTS["logistic_regression"]).T.round(4)

## 6. Baseline 2: XGBoost (early stopping on validation AUC)

In [ ]:
spw = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
xgb_model = xgb.XGBClassifier(
    n_estimators=2000, learning_rate=0.02, max_depth=3, min_child_weight=100, subsample=0.7, colsample_bytree=0.7,
    reg_lambda=10.0, gamma=0.0, scale_pos_weight=spw, eval_metric="auc", early_stopping_rounds=100,
    tree_method="hist", random_state=CFG["SEED"], n_jobs=-1)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
xg_va, xg_te = xgb_model.predict_proba(X_va)[:, 1], xgb_model.predict_proba(X_te)[:, 1]
xg_thr = tune_threshold(y_va, xg_va)
RESULTS["xgboost"] = {"val": metrics(y_va, xg_va, xg_thr), "test": metrics(y_te, xg_te, xg_thr)}
print("best iteration:", xgb_model.best_iteration, "| scale_pos_weight:", round(spw, 3))
pd.DataFrame(RESULTS["xgboost"]).T.round(4)

## 7. Compare against the benchmark
A model only counts if it beats the **majority-class benchmark and 0.5 AUC on TEST**, with a confidence interval that excludes chance.
The CI uses a **moving-block bootstrap over dates** (rows on the same date and neighbouring dates are correlated; 5-day windows overlap).

In [ ]:
rows = []
for model, r in RESULTS.items():
    for split in ("val", "test"):
        rows.append({"model": model, "split": split, **{k: round(v, 4) for k, v in r[split].items() if k not in ("tn", "fp", "fn", "tp")}})
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

def block_bootstrap(df, proba, thr, reps, block, seed):
    rng = np.random.default_rng(seed)
    df = df.reset_index(drop=True); y = df[TARGET].to_numpy()
    by_date = df.groupby("trading_date").indices
    keys = sorted(by_date); n_blocks = int(np.ceil(len(keys) / block))
    aucs, bals = [], []
    for _ in range(reps):
        starts = rng.integers(0, max(len(keys) - block, 1), n_blocks)
        idx = np.concatenate([by_date[k] for s in starts for k in keys[s:s + block]])
        if len(np.unique(y[idx])) < 2: continue
        aucs.append(roc_auc_score(y[idx], proba[idx]))
        bals.append(balanced_accuracy_score(y[idx], (proba[idx] >= thr).astype(int)))
    q = lambda a: tuple(float(x) for x in np.round(np.percentile(a, [2.5, 97.5]), 4))
    return {"auc_ci95": q(aucs), "balanced_acc_ci95": q(bals)}

CI = {"logistic_regression": block_bootstrap(test, lr_te, lr_thr, CFG["BOOTSTRAP_REPS"], CFG["BOOTSTRAP_BLOCK_DATES"], CFG["SEED"]),
      "xgboost": block_bootstrap(test, xg_te, xg_thr, CFG["BOOTSTRAP_REPS"], CFG["BOOTSTRAP_BLOCK_DATES"], CFG["SEED"])}
for m, ci in CI.items():
    RESULTS[m]["test_ci95"] = ci
    print(f"{m:20s} TEST  AUC 95% CI {ci['auc_ci95']}   balanced-acc 95% CI {ci['balanced_acc_ci95']}")

best_auc = max(RESULTS[m]["test"]["roc_auc"] for m in ("logistic_regression", "xgboost"))
if best_auc > 0.65:
    print("\nWARNING: test AUC > 0.65 is unusually high for 5-day direction. Re-check for leakage before trusting it.")
elif best_auc < 0.52:
    print("\nNOTE: test AUC is near chance. That is a legitimate, expected baseline result for 5-day direction on raw OHLCV+RSI/MACD.")

## 8. What the models use (sanity check: no single feature should dominate absurdly)

In [ ]:
imp = pd.Series(xgb_model.get_booster().get_score(importance_type="gain")).reindex(FEATURES).fillna(0).sort_values(ascending=False)
coef = pd.Series(logreg[-1].coef_[0], index=FEATURES).sort_values(key=abs, ascending=False)
print("XGBoost gain importance (normalised):"); print((imp / imp.sum()).round(4).head(10).to_string())
print("\nLogistic regression coefficients (standardised features):"); print(coef.round(4).head(10).to_string())

## 9. Save artifacts
Model selection uses **validation AUC only** (never test). Saved: both models, `feature_list.json`, `metrics.json`, `config.json`, the export's `model_a_meta.json`, plus a zip.

In [ ]:
OUT = Path(CFG["OUT_DIR"])
selected = max(("logistic_regression", "xgboost"), key=lambda m: RESULTS[m]["val"]["roc_auc"])
print("selected by validation AUC:", selected)

xgb_model.save_model(OUT / "model_a_xgboost.json")
joblib.dump(logreg, OUT / "model_a_logreg.joblib")
(OUT / "feature_list.json").write_text(json.dumps(FEATURES, indent=2))

def py(o):
    if isinstance(o, dict): return {k: py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [py(v) for v in o]
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    return o

(OUT / "metrics.json").write_text(json.dumps(py(RESULTS), indent=2))
config = dict(
    task="Model A: 5-session price direction, OHLCV + RSI/MACD", target=TARGET, horizon_sessions=HORIZON, features=FEATURES,
    cfg=CFG, selected_model=selected,
    thresholds={"logistic_regression": lr_thr, "xgboost": xg_thr},
    logreg_preprocessing={"order": "clip -> median impute -> standard scale -> LR", "clip_low": lo.tolist(), "clip_high": hi.tolist()},
    xgboost_params={k: v for k, v in xgb_model.get_params().items() if k in
                    ("n_estimators", "learning_rate", "max_depth", "min_child_weight", "subsample", "colsample_bytree",
                     "reg_lambda", "scale_pos_weight")} | {"best_iteration": int(xgb_model.best_iteration)},
    split={"val_start": str(val_start.date()), "test_start": str(test_start.date()),
           "train_rows": len(train), "val_rows": len(val), "test_rows": len(test)},
    price_basis=meta["price_basis"], prediction_cutoff=meta["prediction_cutoff"],
    versions={"python": platform.python_version(), "pandas": pd.__version__, "sklearn": sklearn.__version__,
              "xgboost": xgb.__version__, "numpy": np.__version__},
)
(OUT / "config.json").write_text(json.dumps(py(config), indent=2))
shutil.copy(DATA_DIR / "model_a_meta.json", OUT / "dataset_meta.json")

# round-trip check: reload from disk and confirm predictions are identical
re_x = xgb.XGBClassifier(); re_x.load_model(OUT / "model_a_xgboost.json")
re_l = joblib.load(OUT / "model_a_logreg.joblib")
assert np.allclose(re_x.predict_proba(X_te[json.loads((OUT / "feature_list.json").read_text())])[:, 1], xg_te, atol=1e-6)
assert np.allclose(re_l.predict_proba(clip(X_te))[:, 1], lr_te, atol=1e-9)
print("round-trip OK. Files:", sorted(p.name for p in OUT.iterdir()))

archive = shutil.make_archive(str(OUT), "zip", OUT)
if CFG["DATA_SOURCE"] == "drive":
    dest = DATA_DIR / "artifacts"; shutil.copytree(OUT, dest, dirs_exist_ok=True); print("copied to Drive:", dest)
elif CFG["DATA_SOURCE"] == "upload":
    from google.colab import files; files.download(archive)
print("zip:", archive)

## 10. How to read the result (honest expectations)
* Five-day direction of liquid equities from price-only features is a **hard** problem; test AUC of roughly 0.50-0.56 is typical and not a bug.
* If both models are inside chance on test (CI includes 0.50), report that as the Model A baseline. It is the benchmark that sentiment/fundamentals must beat.
* If AUC > 0.65, suspect leakage first (stale/partial bars, a feature built from future data).
* Raw closes are unadjusted: cash dividends cause small unflagged drops. Quantify that as a known limitation.